# V9-A — Replication & generalization

V8-A (diagonal gated delta-rule SSM) took the crown at **1.319** vs V7's
1.339 — on a single seed. The 0.020 margin sits near run-to-run noise, so
the crown is provisional until this validation lands. V9-A changes nothing
architectural; it re-runs the campaign's key results:

1. **Multi-seed replication** — 5 seeds x (V8-A + attention), 1500 steps,
   identical config to V8-A Run 1. All campaign gates on every seed.
2. **Dataset generalization** — enwik8 (byte vocab 256) + text8 (27-char
   vocab); the parameter match is re-derived per dataset (embedding rows
   change — the matched-params discipline is re-established, not assumed).
3. **Length generalization** — zero-shot eval of the T=128-trained models at
   T=256/512/1024/2048 (free), plus one token-matched T=512 training run
   (375 steps). An attention OOM where v8a fits is a result, not a failure.

Runtime estimate: ~1.5-2h on a Colab T4 (10 x 1500-step runs at ~5 min each
+ datasets + T=512 + evals).

**Confirm criterion** (brief §1a): mean(v8a) < mean(attn) with margin > 2x
pooled std, AND mean(v8a) < 1.339 - noise. Report mean +/- std for final
val, best val, and the step-250 checkpoint (early-learning claim).

**Overturn criterion**: if the v8a-V7 gap vanishes inside noise, or any seed
flips the v8a>attn ranking -> crown is provisional, V9 quality work is
premature. Dataset ranking flips or faster v8a degradation with T scope the
claims down to the current corpus/length — itself the honest result.


## 0. Setup (+ Triton)

In [ ]:
import torch
import triton
print("torch", torch.__version__)
print("triton", triton.__version__)
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
device = "cuda" if torch.cuda.is_available() else "cpu"
assert torch.cuda.is_available(), "V9-A needs the T4 GPU"

def set_seed(s):
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

import math
def check_finite(tag, vals):
    for v in vals:
        if math.isnan(v) or math.isinf(v):
            print(f"NUMERICAL FAULT at {tag}: non-finite value {v} -- STOPPING", flush=True)
            raise SystemExit(f"numerical fault at {tag}")


torch 2.11.0+cu130
triton 3.6.0
cuda: True Tesla T4


## 1. Data — three corpora (current + enwik8 + text8)

In [ ]:
import os, urllib.request, zipfile
os.makedirs("data", exist_ok=True)

def dl(url, path):
    if not os.path.exists(path):
        print("downloading", path, flush=True)
        urllib.request.urlretrieve(url, path)
    return path

# --- corpus A: the campaign corpus (TinyShakespeare + Gutenberg pg11/84/1342)
URLS = {
    "tinyshakespeare.txt": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "alice.txt":           "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "frankenstein.txt":    "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "pride.txt":           "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
}
parts = []
for name, url in URLS.items():
    p = dl(url, f"data/{name}")
    with open(p, encoding="utf-8", errors="ignore") as f:
        parts.append(f.read())
text_a = "\n".join(parts)
chars_a = sorted(set(text_a)); vocab_a = len(chars_a)
stoi_a = {c: i for i, c in enumerate(chars_a)}
data_a = torch.tensor([stoi_a[c] for c in text_a], dtype=torch.long)
nl_a = stoi_a.get("\n")
assert nl_a is not None, "newline not in vocab A!"

# --- corpus B: enwik8 (byte-level, vocab 256)
p = dl("https://mattmahoney.net/dc/enwik8.zip", "data/enwik8.zip")
with zipfile.ZipFile(p) as z:
    z.extract("enwik8", "data")
raw_b = open("data/enwik8", "rb").read()
data_b = torch.tensor(list(raw_b), dtype=torch.long)  # bytes are the tokens
vocab_b, nl_b = 256, 10  # byte 10 == newline

# --- corpus C: text8 (27-char vocab; no newlines -> word-boundary reset)
p = dl("https://mattmahoney.net/dc/text8.zip", "data/text8.zip")
with zipfile.ZipFile(p) as z:
    z.extract("text8", "data")
text_c = open("data/text8", encoding="utf-8").read()
chars_c = sorted(set(text_c)); vocab_c = len(chars_c)
stoi_c = {c: i for i, c in enumerate(chars_c)}
data_c = torch.tensor([stoi_c[c] for c in text_c], dtype=torch.long)
# text8 has no newlines: use SPACE as the segment boundary so the reset
# mechanism stays active (documented adaptation, not a silent change).
nl_c = stoi_c.get(" ")
print(f"corpus A: {len(text_a)/1e6:.2f} MB vocab {vocab_a} nl_id={nl_a}")
print(f"corpus B (enwik8): {len(raw_b)/1e6:.2f} MB vocab {vocab_b} nl_id={nl_b}")
print(f"corpus C (text8): {len(text_c)/1e6:.2f} MB vocab {vocab_c} chars={''.join(chars_c)!r} boundary(space)_id={nl_c}")
assert vocab_c == 27, f"text8 vocab drift: {vocab_c}"

def split95(d):
    n = int(0.95 * len(d))
    return d[:n], d[n:]

DATA = {
    "A-corpus": (data_a, vocab_a, nl_a),
    "B-enwik8": (data_b, vocab_b, nl_b),
    "C-text8":  (data_c, vocab_c, nl_c),
}

def make_get_batch(d, seq_len, batch_size):
    def get_batch(split="train"):
        dd = d[:int(0.95 * len(d))] if split == "train" else d[int(0.95 * len(d)):]
        i = torch.randint(0, len(dd) - seq_len - 1, (batch_size,))
        x = torch.stack([dd[j:j+seq_len] for j in i]).to(device)
        y = torch.stack([dd[j+1:j+seq_len+1] for j in i]).to(device)
        return x, y
    return get_batch


downloading data/tinyshakespeare.txt
downloading data/alice.txt
downloading data/frankenstein.txt
downloading data/pride.txt
downloading data/enwik8.zip
downloading data/text8.zip
corpus A: 2.47 MB vocab 104 nl_id=0
corpus B (enwik8): 100.00 MB vocab 256 nl_id=10
corpus C (text8): 100.00 MB vocab 27 chars=' abcdefghijklmnopqrstuvwxyz' boundary(space)_id=0


## 2. Models + Triton kernels (inlined, no imports)

In [ ]:
"""
Linear-recurrent replacement for Transformer self-attention.

Experiment: take a standard decoder-only Transformer, rip out the
causal self-attention layer, and replace it with a *moving linear
equation* that compresses the past into a fixed-size running state:

    h_t = d * h_{t-1} + (1 - d) * (B x_t)        (linear recurrence)
    y_t = C h_t                                   (readout)

where d in (0,1)^S is a learned per-channel decay, B/C are learned
projections, and h has FIXED size S independent of sequence length.

This is O(T) time and O(1) memory w.r.t. context length, vs O(T^2)
for attention. The question: how much modeling power do we lose?
"""

import math
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Token mixing layers
# ----------------------------------------------------------------------------

class SelectiveSegmentedState(nn.Module):
    """V5: gated segmented state-space mixer.

    Replaces the static decay with two dynamic mechanisms:

    1. Dynamic Selective Gate (the model's "will"):
         g_t = sigmoid(W_g x_t + b_g)      in (0,1), per token & channel
       acts as a learned per-token filter: low-information tokens get
       g_t -> 0 (forgotten fast), salient tokens get g_t -> 1 (kept).

    2. Hard Memory Reset Mask (segment boundaries):
         r_t = 1 if token t is a structural boundary (e.g. newline)
         h_t = (1 - r_t) * (g_t * h_{t-1} + (1 - g_t) * u_t)
       when r_t = 1 the state is multiplied by zero: the past memory
       buffer is completely destroyed and the new segment starts fresh.
       (The residual stream still carries x_t itself forward.)

    The recurrence stays affine in h (h_t = a_t*h_{t-1} + b_t with
    a_t = (1-r_t)*g_t, b_t = (1-r_t)*(1-g_t)*u_t), so the parallel
    associative scan still applies: O(T) work, log2(T) parallel passes.
    State dim stays frozen at `state_dim` (VRAM edge preserved).
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)  # bias = per-channel prior
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        # x: (B,T,dim); reset: (B,T) binary, 1 = boundary token
        u = self.in_proj(x)                    # (B,T,S)
        g = torch.sigmoid(self.gate_proj(x))   # (B,T,S) dynamic gate
        if reset is not None:
            keep = (1.0 - reset).unsqueeze(-1)  # (B,T,1)
            a = keep * g
            b = keep * (1.0 - g) * u
        else:
            a = g
            b = (1.0 - g) * u
        h_seq = parallel_scan_ab(a, b)          # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention (the baseline)."""
    def __init__(self, dim, n_heads, dropout=0.0):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                       # (B,T,H,D)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B,H,T,D)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)


def parallel_scan_ab(A, B):
    """Parallel associative scan (Hillis-Steele) for h_t = A_t*h_{t-1} + B_t.

    A, B: (B, T, S) per-step coefficients. Returns H: (B, T, S).
    log2(T) fully-parallel passes instead of T sequential steps.
    """
    Bsz, T, S = A.shape
    A = A.contiguous()
    Bh = B.contiguous()
    idx = torch.arange(T, device=A.device).view(1, T, 1)
    o = 1
    while o < T:
        m = idx >= o
        A_sh = torch.roll(A, o, dims=1)
        B_sh = torch.roll(Bh, o, dims=1)
        # compose segment [t-o, t-1] then [t]:
        #   a_new = a_t * a_{t-o},  b_new = a_t * b_{t-o} + b_t
        A_new = A * A_sh
        B_new = A * B_sh + Bh
        A = torch.where(m, A_new, A)
        Bh = torch.where(m, B_new, Bh)
        o *= 2
    return Bh


def parallel_linear_recurrence(u, decay):
    """Parallel associative scan (Hillis-Steele) for h_t = d*h_{t-1} + c_t.

    Reformulates the sequential loop as a tree reduction:
      - each step is an affine map  h -> a*h + b, packed as pair (a, b)
      - pair composition is associative:
            (a1,b1) then (a2,b2)  =  (a2*a1, a2*b1 + b2)
      - a parallel prefix over pairs yields every h_t at once

    log2(T) fully-parallel passes instead of T sequential steps.
    u: (B, T, S) inputs already projected; decay: (S,) in (0, 1).
    Returns h: (B, T, S) — identical math to the for-loop version.
    """
    Bsz, T, S = u.shape
    c = (1.0 - decay) * u
    A = decay.expand(Bsz, T, S).contiguous()   # a-coefficients
    return parallel_scan_ab(A, c)


class LinearRunningState(nn.Module):
    """Attention replacement: fixed-size running state, linear recurrence.

    h_t = decay * h_{t-1} + (1 - decay) * (B x_t)
    y_t = C h_t

    - decay: learned per-state-channel forget factor in (0, 1)
    - B:     input  projection  dim -> state_dim
    - C:     output projection  state_dim -> dim
    State size is FIXED (does not grow with context length).
    mode="loop": native Python for-loop (sequential, slow on GPU)
    mode="scan": parallel associative scan (log2(T) parallel passes)
    """
    def __init__(self, dim, state_dim=None, dropout=0.0, mode="loop"):
        super().__init__()
        self.mode = mode
        self.state_dim = state_dim or dim
        # init decay ~ U(0.9, 0.999) -> long-ish memory at start
        init = torch.empty(self.state_dim).uniform_(math.log(0.9 / 0.1),
                                                   math.log(0.999 / 0.001))
        self.logit_decay = nn.Parameter(init)
        self.in_proj = nn.Linear(dim, self.state_dim, bias=False)
        self.out_proj = nn.Linear(self.state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, _ = x.shape
        decay = torch.sigmoid(self.logit_decay)          # (S,) in (0,1)
        u = self.in_proj(x)                              # (B,T,S)
        if self.mode == "scan":
            h_seq = parallel_linear_recurrence(u, decay)  # (B,T,S)
        else:
            keep = 1.0 - decay
            h = torch.zeros(B, self.state_dim, device=x.device, dtype=x.dtype)
            outs = []
            for t in range(T):
                h = decay * h + keep * u[:, t]
                outs.append(h)
            h_seq = torch.stack(outs, dim=1)              # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


# ----------------------------------------------------------------------------
# Blocks & models (identical except for the mixing layer)
# ----------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self, dim, n_heads, mixer, dropout=0.0, ffn_hidden=None):
        super().__init__()
        ffn_hidden = ffn_hidden or 4 * dim
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.mix = mixer
        self.mlp = nn.Sequential(
            nn.Linear(dim, ffn_hidden), nn.GELU(),
            nn.Linear(ffn_hidden, dim), nn.Dropout(dropout))

    def forward(self, x, reset=None):
        x = x + self.mix(self.ln1(x), reset)
        x = x + self.mlp(self.ln2(x))
        return x


class TinyLM(nn.Module):
    """Decoder-only LM. mixer_fn(dim, n_heads) builds the token-mixing layer."""
    def __init__(self, vocab, dim=128, n_layers=4, n_heads=4,
                 seq_len=128, mixer_fn=None, dropout=0.0, ffn_hidden=None,
                 newline_id=None):
        super().__init__()
        self.seq_len = seq_len
        self.newline_id = newline_id  # boundary token id for reset mask
        self.tok_emb = nn.Embedding(vocab, dim)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), dropout, ffn_hidden)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(
            torch.arange(T, device=idx.device))[None]
        reset = None
        if self.newline_id is not None:
            reset = (idx == self.newline_id).float()  # (B,T) boundary mask
        for blk in self.blocks:
            x = blk(x, reset)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new, temp=1.0, top_k=None):
        for _ in range(max_new):
            ctx = idx[:, -self.seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temp, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_models(vocab, dim=128, n_layers=4, n_heads=4, seq_len=128,
                 state_dim=None, dropout=0.0):
    attn = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: CausalSelfAttention(d, h, dropout),
                  dropout=dropout)
    lin = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                 mixer_fn=lambda d, h: LinearRunningState(d, state_dim, dropout),
                 dropout=dropout)
    return attn, lin


"""(inlined) V6/V7/V8-A: Triton-fused scans.

Fuses the V5 recurrence (per the V6 execution plan):
  Step A: the 256-wide hidden state lives in SRAM registers for the whole
          sequence -- loaded once per batch element, never spilled to HBM.
  Step B: sigmoid gate, hard reset mask, and the linear update
          h = keep*g*h + keep*(1-g)*u run in registers, every timestep.
  Step C: only h_t (plus a_t, saved for the backward pass) hit HBM.
          The parallel scan's O(log T) intermediate (B,T,S) tensors vanish.

Math is IDENTICAL to V5's SelectiveSegmentedState -- only the execution
path changes. Zero new parameters.
"""
import triton
import triton.language as tl

BLOCK_S = 256  # state dim; the whole state fits in SRAM


@triton.jit
def _sel_scan_fwd_kernel(u_ptr, gp_ptr, keep_ptr, h_ptr, a_ptr,
                         T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with a_t = keep_t*g_t,
    b_t = keep_t*(1-g_t)*u_t, g_t = sigmoid(gp_t)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # Step A: state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))              # Step B: gate in registers
        at = k * g
        h = at * h + k * (1.0 - g) * u_t             # Step B: update in registers
        tl.store(h_b + t * BLOCK + offs, h)          # Step C: outputs to HBM
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _sel_scan_bwd_kernel(dh_ptr, hs_ptr, ap_ptr, u_ptr, gp_ptr, keep_ptr,
                         du_ptr, dgp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    ap is (B,T+1,S) with ap[t+1] = a_{t+1} (ap[T] = 0).
    D_t = gout_t + a_{t+1}*D_{t+1}; du_t = k(1-g)D_t;
    dgp_t = k*g*(1-g)*(h_{t-1}-u_t)*D_t."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    ap_b = ap_ptr + b * bsp
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dgp_b = dgp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(ap_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))
        omg = 1.0 - g
        tl.store(du_b + t * BLOCK + offs, k * omg * D)
        tl.store(dgp_b + t * BLOCK + offs, k * g * omg * (h_prev - u_t) * D)
        dnext = D


class _SelScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, gp, keep):
        assert u.is_cuda and u.dtype == torch.float32, "V6 kernel: CUDA fp32"
        assert u.shape == gp.shape and u.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, gp, keep = u.contiguous(), gp.contiguous(), keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _sel_scan_fwd_kernel[(B,)](u, gp, keep, h, a, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, gp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, gp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h    # hs[t] = h_{t-1}
        ap = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        ap[:, :T] = a    # ap[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dgp = torch.empty_like(gp)
        _sel_scan_bwd_kernel[(B,)](dh, hs, ap, u, gp, keep, du, dgp, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        return du, dgp, None


def triton_selective_scan(u, gp, keep):
    """Fused selective scan. u, gp: (B,T,256) fp32 CUDA; keep: (B,T) fp32.
    Returns h: (B,T,256). Differentiable."""
    return _SelScanFn.apply(u, gp, keep)


class SelectiveSegmentedStateTriton(nn.Module):
    """V6 mixer: IDENTICAL math and parameters to V5's SelectiveSegmentedState.

    The only change is execution: the recurrence runs in the fused Triton
    kernel (Step A/B/C) instead of the PyTorch Hillis-Steele scan.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V6 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = self.gate_proj(x).float().contiguous()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)
        return self.dropout(self.out_proj(h_seq.to(x.dtype)))


class SelectiveSegmentedStateV7(nn.Module):
    """V7 mixer: V6's Triton-fused selective scan + two literature grafts.

    Graft #1 — output gate + state normalization (HGRN2 / Mamba-2 / GDN):
        y_t = C · (o_t ⊙ RMSNorm(h_t)),   o_t = σ(W_o x_t + b_o)
    States accumulate over segments so readout scale drifts; every strong
    post-Mamba block gates the readout. RMSNorm here is parameter-free
    (no scale vector) to keep the parameter budget exact.

    Graft #3 — layer-graded forget floor (HGRN, NeurIPS'23):
        g_t = σ(W_g x_t + b_g + floor_l)
    floor_l is a per-layer learnable scalar, initialized monotonically
    increasing bottom→top, so low layers forget fast (spelling-level) and
    top layers retain (phrase-level). Complements the newline hard reset.

    The recurrence math is UNCHANGED → the same fused Triton kernel as V6.
    With FFN 1023 the budget lands EXACTLY on the attention baseline.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V7 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)          # (B,T,S)
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


# ----------------------------------------------------------------------------
# V8-A: diagonal gated delta-rule scan
# ----------------------------------------------------------------------------

@triton.jit
def _delta_scan_fwd_kernel(u_ptr, ap_ptr, bp_ptr, kp_ptr, keep_ptr,
                           h_ptr, a_ptr, T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with
        alpha = sigmoid(ap_t), beta = sigmoid(bp_t), kk = sigmoid(kp_t),
        v = silu(u_t),
        a_t = keep_t * alpha * (1 - beta*kk^2),
        b_t = keep_t * beta * kk * v.
    Diagonal restriction of Gated DeltaNet's S_t = a_t S_{t-1}(I-b_t k k^T)
    + b_t v k^T to per-channel 1x1 systems. Strictly contractive:
    sigmoid bounds keep (1-beta*kk^2) in (0,1)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su                                 # SiLU in registers
        at = kr * alpha * (1.0 - beta * kk * kk)
        bt = kr * beta * kk * v
        h = at * h + bt
        tl.store(h_b + t * BLOCK + offs, h)
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _delta_scan_bwd_kernel(dh_ptr, hs_ptr, apn_ptr, u_ptr, ap_ptr, bp_ptr,
                           kp_ptr, keep_ptr, du_ptr, dap_ptr, dbp_ptr,
                           dkp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    apn is (B,T+1,S) with apn[t+1] = a_{t+1} (apn[T] = 0).
    Adjoint: D_t = gout_t + a_{t+1}*D_{t+1}; then per-step
      du  = D*kr*beta*kk*silu'(u)
      dap = D*h_prev*kr*alpha*(1-alpha)*(1-beta*kk^2)
      dbp = D*kr*beta*(1-beta)*(kk*v - alpha*kk^2*h_prev)
      dkp = D*kr*beta*kk*(1-kk)*(v - 2*alpha*kk*h_prev)
    (hand-derived; verified against autograd on CPU before transcription)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    apn_b = apn_ptr + b * bsp
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dap_b = dap_ptr + b * bs
    dbp_b = dbp_ptr + b * bs
    dkp_b = dkp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(apn_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su
        silu_p = su * (1.0 + u_t * (1.0 - su))
        k2 = kk * kk
        omb = 1.0 - beta * k2
        tl.store(du_b + t * BLOCK + offs,
                 D * kr * beta * kk * silu_p)
        tl.store(dap_b + t * BLOCK + offs,
                 D * h_prev * kr * alpha * (1.0 - alpha) * omb)
        tl.store(dbp_b + t * BLOCK + offs,
                 D * kr * beta * (1.0 - beta) * (kk * v - alpha * k2 * h_prev))
        tl.store(dkp_b + t * BLOCK + offs,
                 D * kr * beta * kk * (1.0 - kk) * (v - 2.0 * alpha * kk * h_prev))
        dnext = D


def _delta_scan_cpu(u, ap, bp, kp, keep):
    """CPU reference for the delta scan: differentiable pure-PyTorch loop.

    Used ONLY for local (CPU) verification. On CUDA the fused Triton
    kernel above is used instead (gate-checked against this math on Colab).
    """
    B, T, S = u.shape
    kr = keep.unsqueeze(-1)
    alpha = torch.sigmoid(ap)
    beta = torch.sigmoid(bp)
    kk = torch.sigmoid(kp)
    v = u * torch.sigmoid(u)                          # SiLU
    h = torch.zeros(B, S, dtype=u.dtype, device=u.device)
    hs = []
    for t in range(T):
        a = kr[:, t] * alpha[:, t] * (1.0 - beta[:, t] * kk[:, t] ** 2)
        bb = kr[:, t] * beta[:, t] * kk[:, t] * v[:, t]
        h = a * h + bb
        hs.append(h)
    return torch.stack(hs, dim=1)


class _DeltaScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, ap, bp, kp, keep):
        assert u.is_cuda and u.dtype == torch.float32, \
            "V8-A kernel: CUDA fp32"
        for t in (u, ap, bp, kp):
            assert t.shape == u.shape and t.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, ap, bp, kp = u.contiguous(), ap.contiguous(), bp.contiguous(), kp.contiguous()
        keep = keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _delta_scan_fwd_kernel[(B,)](u, ap, bp, kp, keep, h, a, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, ap, bp, kp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, ap, bp, kp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h      # hs[t] = h_{t-1}
        apn = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        apn[:, :T] = a     # apn[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dap = torch.empty_like(ap)
        dbp = torch.empty_like(bp)
        dkp = torch.empty_like(kp)
        _delta_scan_bwd_kernel[(B,)](dh, hs, apn, u, ap, bp, kp, keep,
                                    du, dap, dbp, dkp, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        return du, dap, dbp, dkp, None


def triton_delta_scan(u, ap, bp, kp, keep):
    """Fused diagonal delta-rule scan.

    u: raw Bx (SiLU applied in-kernel); ap/bp/kp: pre-sigmoid
    alpha/beta/key projections; keep: (B,T) reset mask.
    (B,T,256) fp32; CUDA -> Triton kernel, CPU -> differentiable
    reference loop (verification only). Returns h: (B,T,256).
    """
    if u.is_cuda:
        return _DeltaScanFn.apply(u, ap, bp, kp, keep)
    return _delta_scan_cpu(u, ap, bp, kp, keep)


class SelectiveSegmentedStateV8A(nn.Module):
    """V8-A mixer: diagonal gated delta rule on V7's chassis.

    Per channel (delta rule = erase-before-write, Gated DeltaNet Eq.10
    restricted to 1x1 systems):
        alpha_t = sigmoid(W_a x_t + b_a + floor_l)   # reuse gate_proj+floor
        beta_t  = sigmoid(W_b x_t + b_b),  b_b = 0   # NEW write_proj
        k_t     = sigmoid(W_k x_t + b_k),  b_k = +2  # NEW key_proj
        v_t     = SiLU(B x_t)                        # reuse in_proj + SiLU
        a_t = alpha_t * (1 - beta_t * k_t^2)
        b_t = beta_t * k_t * v_t
        h_t = (1 - r_t) * (a_t .* h_{t-1} + b_t)
        y_t = C . (o_t .* RMSNorm(h_t)), o_t = sigmoid(W_o x_t)

    Init puts the model near V7's convex-blend regime (k~=0.88, beta~=0.5)
    and the sigmoid bounds keep every step strictly contractive.
    With FFN 766 the budget lands within the relaxed <=3000 gate.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V8-A kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)   # B
        self.gate_proj = nn.Linear(dim, state_dim)             # W_alpha
        self.write_proj = nn.Linear(dim, state_dim)            # W_beta (NEW)
        self.key_proj = nn.Linear(dim, state_dim)              # W_k (NEW)
        self.out_gate_proj = nn.Linear(dim, state_dim)         # W_o
        self.out_proj = nn.Linear(state_dim, dim, bias=False)  # C
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.write_proj.bias.data.zero_()       # b_beta = 0  -> beta ~= 0.5
        self.key_proj.bias.data.fill_(2.0)       # b_k = +2.0  -> k ~= 0.88
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6
        self.last_state = None  # stashed h_seq for stability logging (no grad)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        ap = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        bp = self.write_proj(x).float().contiguous()
        kp = self.key_proj(x).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_delta_scan(u, ap, bp, kp, keep)      # (B,T,S)
        self.last_state = h_seq.detach()  # stability logging only; no numerics change
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


## 3. KERNEL GATE — Triton delta kernel vs reference (HARD STOP on fail)

In [ ]:
# The kernel is seed-independent: run the gate ONCE, hard-STOP the run if it fails.
set_seed(0)
B, T, S = 4, 128, 256
u = torch.randn(B, T, S, device=device)
ap = torch.randn(B, T, S, device=device)
bp = torch.randn(B, T, S, device=device)
kp = torch.randn(B, T, S, device=device)
keep = (torch.rand(B, T, device=device) > 0.3).float()

h_tri = triton_delta_scan(u, ap, bp, kp, keep)  # compiles on first call

kr = keep.unsqueeze(-1)
alpha = torch.sigmoid(ap); beta = torch.sigmoid(bp); kk = torch.sigmoid(kp)
v = u * torch.sigmoid(u)
a = kr * alpha * (1 - beta * kk * kk)
b = kr * beta * kk * v
h_scan = parallel_scan_ab(a, b)

h = torch.zeros(B, S, device=device); hs = []
for t in range(T):
    at = kr[:, t] * alpha[:, t] * (1 - beta[:, t] * kk[:, t] ** 2)
    bt = kr[:, t] * beta[:, t] * kk[:, t] * v[:, t]
    h = at * h + bt
    hs.append(h)
h_loop = torch.stack(hs, 1)

e_scan = float((h_tri - h_scan).abs().max())
e_loop = float((h_tri - h_loop).abs().max())
print(f"triton vs delta scan max delta: {e_scan:.2e}")
print(f"triton vs delta loop max delta: {e_loop:.2e}")

u2 = u.clone().requires_grad_(True); ap2 = ap.clone().requires_grad_(True)
bp2 = bp.clone().requires_grad_(True); kp2 = kp.clone().requires_grad_(True)
triton_delta_scan(u2, ap2, bp2, kp2, keep).pow(2).sum().backward()
u3 = u.clone().requires_grad_(True); ap3 = ap.clone().requires_grad_(True)
bp3 = bp.clone().requires_grad_(True); kp3 = kp.clone().requires_grad_(True)
al3 = torch.sigmoid(ap3); be3 = torch.sigmoid(bp3); kk3 = torch.sigmoid(kp3)
v3 = u3 * torch.sigmoid(u3)
a3 = kr * al3 * (1 - be3 * kk3 * kk3)
b3 = kr * be3 * kk3 * v3
parallel_scan_ab(a3, b3).pow(2).sum().backward()
errs = {}
for name, g2, g3 in [("du", u2.grad, u3.grad), ("dap", ap2.grad, ap3.grad),
                     ("dbp", bp2.grad, bp3.grad), ("dkp", kp2.grad, kp3.grad)]:
    errs[name] = float((g2 - g3).abs().max())
print("backward max deltas:", {k: f"{v:.2e}" for k, v in errs.items()})
if not (e_scan < 1e-6 and e_loop < 1e-6):
    raise SystemExit("KERNEL GATE FAILED (forward) -- STOPPING RUN")
if not all(v < 1e-5 for v in errs.values()):
    raise SystemExit("KERNEL GATE FAILED (backward) -- STOPPING RUN")
print("KERNEL GATE PASSED — fused delta kernel == reference math")


triton vs delta scan max delta: 3.58e-07
triton vs delta loop max delta: 3.58e-07
backward max deltas: {'du': '1.43e-06', 'dap': '6.56e-07', 'dbp': '9.54e-07', 'dkp': '7.15e-07'}
KERNEL GATE PASSED — fused delta kernel == reference math


## 4. Param-diff gate per dataset (HARD STOP if diff > 3000)

In [ ]:
# The matched-params discipline is re-established PER DATASET, not assumed:
# embedding rows change with vocab, so the FFN width is re-solved for each
# corpus and the counted diff is verified by construction.
def count_params(m):
    return sum(p.numel() for p in m.parameters())

DIM, LAYERS, HEADS, SEQ_MAX = 256, 8, 8, 2048
# SEQ_MAX=2048 at construction so the zero-shot length eval (T up to 2048)
# needs no rebuild; training still uses T=128 windows. pos_emb rows are
# identical in both models, so the diff is unaffected.

def solve_ffn(vocab, newline_id):
    # Measure the diff once with both models at FFN 1024, then solve the width
    # analytically: per-block FFN params are exactly 513h+256 (Linear(256->h)
    # + Linear(h->256), both biased), so diff(h) = C + 8*513*h with C measured.
    # The slope is structural from the Block code; the intercept is measured,
    # not assumed.
    set_seed(0)
    a = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_MAX,
               mixer_fn=lambda d, h: CausalSelfAttention(d, h),
               newline_id=newline_id).to(device)
    v = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_MAX,
               mixer_fn=lambda d, h: SelectiveSegmentedStateV8A(d, state_dim=256),
               ffn_hidden=1024, newline_id=newline_id).to(device)
    d1024 = count_params(a) - count_params(v)
    del a, v
    torch.cuda.empty_cache()
    h_star = int(round(1024 + d1024 / (8 * 513)))
    return h_star

FFN = {}
for dname, (d, vocab, nl_id) in DATA.items():
    h = solve_ffn(vocab, nl_id)
    # verify by construction on real tensors
    set_seed(0)
    attn_m = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_MAX,
                    mixer_fn=lambda d_, h_: CausalSelfAttention(d_, h_),
                    newline_id=nl_id).to(device)
    v8a_m = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_MAX,
                   mixer_fn=lambda d_, h_: SelectiveSegmentedStateV8A(d_, state_dim=256),
                   ffn_hidden=h, newline_id=nl_id).to(device)
    pa_c, pv_c = count_params(attn_m), count_params(v8a_m)
    diff = abs(pa_c - pv_c)
    print(f"{dname}: vocab={vocab} ffn_hidden={h} attention={pa_c} v8a={pv_c} diff={diff}")
    if diff > 3000:
        raise SystemExit(f"PARAM-DIFF GATE FAILED on {dname}: diff {diff} > 3000 -- STOPPING RUN")
    FFN[dname] = h
    del attn_m, v8a_m
    torch.cuda.empty_cache()
print("PARAM-DIFF GATE PASSED on all datasets (diff <= 3000)")

def build_models(vocab, newline_id, ffn_hidden, seed):
    set_seed(seed)
    attention = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_MAX,
        mixer_fn=lambda d, h: CausalSelfAttention(d, h),
        newline_id=newline_id).to(device)
    v8a = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_MAX,
        mixer_fn=lambda d, h: SelectiveSegmentedStateV8A(d, state_dim=256),
        ffn_hidden=ffn_hidden, newline_id=newline_id).to(device)
    _floors = torch.linspace(math.log(0.3/0.7), math.log(0.9/0.1), LAYERS)
    for _i, _blk in enumerate(v8a.blocks):
        _blk.mix.forget_floor.data.fill_(_floors[_i])
    return {"attention": attention, "v8a-delta": v8a}


A-corpus: vocab=104 ffn_hidden=767 attention=6861312 v8a=6863360 diff=2048
B-enwik8: vocab=256 ffn_hidden=767 attention=6900224 v8a=6902272 diff=2048
C-text8: vocab=27 ffn_hidden=767 attention=6841600 v8a=6843648 diff=2048
PARAM-DIFF GATE PASSED on all datasets (diff <= 3000)


## 5. Multi-seed replication (seeds 0-4, V8-A + attention, 1500 steps)

In [ ]:
import time

SEEDS = range(5)
STEPS, EVAL, LR, BS, SEQ = 1500, 250, 3e-4, 32, 128
dA, vocabA, nlA = DATA["A-corpus"]
get_batch = make_get_batch(dA, SEQ, BS)
CKPT_STEPS = [1] + list(range(EVAL, STEPS + 1, EVAL))

def val_loss(m, gb, n=10):
    m.eval()
    with torch.no_grad():
        return sum(float(m(*gb("val"))[1]) for _ in range(n)) / n

def state_rms_norms(m):
    norms = []
    for blk in m.blocks:
        h = getattr(getattr(blk, 'mix', None), 'last_state', None)
        if h is not None:
            norms.append(float(torch.sqrt((h.float() ** 2).mean()).item()))
    return norms

def gate_stats(m, xb):
    # layer-averaged sigmoid gate means; alpha includes the forget floor.
    # attention has no gate_proj -> returns {} and is skipped.
    acc = {"alpha": [], "beta": [], "k": []}
    hooks = []
    for blk in m.blocks:
        mix = blk.mix
        if not hasattr(mix, "gate_proj"):
            continue
        def mk_alpha(mod, inp, out, _mix=mix):
            acc["alpha"].append(float(torch.sigmoid(out.float() + _mix.forget_floor).mean()))
        def mk_beta(mod, inp, out):
            acc["beta"].append(float(torch.sigmoid(out.float()).mean()))
        def mk_k(mod, inp, out):
            acc["k"].append(float(torch.sigmoid(out.float()).mean()))
        hooks += [mix.gate_proj.register_forward_hook(mk_alpha),
                  mix.write_proj.register_forward_hook(mk_beta),
                  mix.key_proj.register_forward_hook(mk_k)]
    was_training = m.training
    m.eval()
    with torch.no_grad():
        m(xb)
    for h_ in hooks:
        h_.remove()
    if was_training:
        m.train()
    return {k: sum(v) / len(v) for k, v in acc.items()} if acc["alpha"] else {}

def probe(m, gb):
    # lightweight post-train probe: NO optimizer steps (V4's re-eval corrupted
    # numbers by training inside the metrics cell -- never again).
    m.eval()
    with torch.no_grad():
        tloss = sum(float(m(*gb("train"))[1]) for _ in range(20)) / 20
        xb, _ = gb("train")
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(50):
            m(xb)
        torch.cuda.synchronize()
        tps = BS * SEQ / ((time.time() - t0) / 50)
    m.train()
    torch.cuda.reset_peak_memory_stats()
    xb, yb = gb("train")
    _, loss = m(xb, yb); m.zero_grad(); loss.backward()
    peak = torch.cuda.max_memory_allocated() / 1e6
    torch.cuda.empty_cache()
    return tloss, tps, peak

results = {}
for seed in SEEDS:
    print(f"===== SEED {seed} =====", flush=True)
    models = build_models(vocabA, nlA, FFN["A-corpus"], seed)
    opt = {n: torch.optim.AdamW(m.parameters(), lr=LR) for n, m in models.items()}
    val_hist = {n: [] for n in models}
    state_hist = {n: [] for n in models}
    grad_hist = {n: [] for n in models}
    last_gn = {}
    for step in range(1, STEPS + 1):
        for name, m in models.items():
            m.train()
            xb, yb = get_batch("train")
            _, loss = m(xb, yb)
            opt[name].zero_grad(); loss.backward()
            last_gn[name] = float(torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0))
            opt[name].step()
        if step % EVAL == 0 or step == 1:
            msg = f"step {step:5d}"
            for name, m in models.items():
                vl = val_loss(m, get_batch)
                val_hist[name].append(vl)
                sn = state_rms_norms(m)
                if sn:
                    state_hist[name].append((min(sn), sum(sn) / len(sn), max(sn)))
                grad_hist[name].append(last_gn[name])
                check_finite(f"seed {seed} step {step} {name} val_loss", [vl])
                check_finite(f"seed {seed} step {step} {name} state_norms", sn)
                check_finite(f"seed {seed} step {step} {name} grad_norm", [last_gn[name]])
                sn_str = f" state_rms[{min(sn):.2f}/{sum(sn)/len(sn):.2f}/{max(sn):.2f}]" if sn else ""
                xb0, _ = get_batch("val")
                gs = gate_stats(m, xb0)
                gs_str = (f" gates[a={gs['alpha']:.3f}/b={gs['beta']:.3f}/k={gs['k']:.3f}]"
                          if gs else "")
                msg += (f" | {name}: val {vl:.3f} (ppl {math.exp(vl):.1f})"
                        f"{sn_str} grad_norm {last_gn[name]:.2f}{gs_str}")
            print(msg, flush=True)
    seed_res = {}
    for name, m in models.items():
        tloss, tps, peak = probe(m, get_batch)
        bi = min(range(len(val_hist[name])), key=lambda i: val_hist[name][i])
        seed_res[name] = {
            "final_val": val_hist[name][-1],
            "best_val": val_hist[name][bi],
            "best_step": CKPT_STEPS[bi],
            "step250_val": val_hist[name][1],
            "train_loss": tloss, "tok_s": tps, "vram_mb": peak,
            "val_hist": list(val_hist[name]),
        }
        print(f"seed {seed} {name}: final {val_hist[name][-1]:.3f} "
              f"best {val_hist[name][bi]:.3f}@{CKPT_STEPS[bi]} "
              f"train_loss {tloss:.3f} tok/s {tps:.0f} vram {peak:.0f}MB", flush=True)
    results[seed] = seed_res
    if seed == 0:
        torch.save(models["v8a-delta"].state_dict(), "/content/v9a_seed0_v8a.pt")
        torch.save(models["attention"].state_dict(), "/content/v9a_seed0_attn.pt")
        print("seed-0 weights saved for zero-shot eval")
    del models, opt
    torch.cuda.empty_cache()
print("MULTI-SEED REPLICATION DONE")


===== SEED 0 =====
step     1 | attention: val 4.102 (ppl 60.5) grad_norm 7.79 | v8a-delta: val 3.982 (ppl 53.6) state_rms[0.08/0.11/0.15] grad_norm 5.17 gates[a=0.632/b=0.501/k=0.877]
step   250 | attention: val 2.396 (ppl 11.0) grad_norm 0.98 | v8a-delta: val 1.837 (ppl 6.3) state_rms[0.13/0.14/0.17] grad_norm 1.02 gates[a=0.612/b=0.487/k=0.847]
step   500 | attention: val 1.991 (ppl 7.3) grad_norm 1.12 | v8a-delta: val 1.578 (ppl 4.8) state_rms[0.14/0.15/0.20] grad_norm 1.01 gates[a=0.593/b=0.468/k=0.822]
step   750 | attention: val 1.729 (ppl 5.6) grad_norm 1.11 | v8a-delta: val 1.470 (ppl 4.4) state_rms[0.14/0.16/0.22] grad_norm 0.97 gates[a=0.580/b=0.456/k=0.805]
step  1000 | attention: val 1.573 (ppl 4.8) grad_norm 1.07 | v8a-delta: val 1.393 (ppl 4.0) state_rms[0.15/0.18/0.24] grad_norm 0.86 gates[a=0.572/b=0.446/k=0.792]
step  1250 | attention: val 1.476 (ppl 4.4) grad_norm 1.01 | v8a-delta: val 1.363 (ppl 3.9) state_rms[0.15/0.18/0.26] grad_norm 0.72 gates[a=0.563/b=0.438/k=0

## 6. Dataset generalization (enwik8 + text8, V8-A + attention, seed 0)

In [ ]:
def train_pair(models, gb, steps, tag):
    # compact training loop with the campaign watch-items: pre-clip grad-norm
    # logging (clip 1.0 unchanged), NaN/Inf SystemExit guard, best-val tracking.
    opt = {n: torch.optim.AdamW(m.parameters(), lr=3e-4) for n, m in models.items()}
    val_hist = {n: [] for n in models}
    last_gn = {}
    for step in range(1, steps + 1):
        for name, m in models.items():
            m.train()
            xb, yb = gb("train")
            _, loss = m(xb, yb)
            opt[name].zero_grad(); loss.backward()
            last_gn[name] = float(torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0))
            opt[name].step()
        if step % EVAL == 0 or step == 1:
            msg = f"[{tag}] step {step:5d}"
            for name, m in models.items():
                vl = val_loss(m, gb)
                val_hist[name].append(vl)
                check_finite(f"{tag} step {step} {name}", [vl, last_gn[name]])
                msg += f" | {name}: {vl:.3f} (ppl {math.exp(vl):.1f}) gnorm {last_gn[name]:.2f}"
            print(msg, flush=True)
    return val_hist

ds_results = {}
for dname in ("B-enwik8", "C-text8"):
    d, vocab, nl_id = DATA[dname]
    print(f"===== {dname} (seed 0) =====", flush=True)
    gb = make_get_batch(d, SEQ, BS)
    models = build_models(vocab, nl_id, FFN[dname], seed=0)
    val_hist = train_pair(models, gb, STEPS, dname)
    dh = {}
    for name in models:
        bi = min(range(len(val_hist[name])), key=lambda i: val_hist[name][i])
        dh[name] = {"final_val": val_hist[name][-1], "best_val": val_hist[name][bi],
                    "best_step": CKPT_STEPS[bi], "step250_val": val_hist[name][1]}
        print(f"[{dname}] {name}: final {val_hist[name][-1]:.3f} "
              f"best {val_hist[name][bi]:.3f}@{CKPT_STEPS[bi]}", flush=True)
    ds_results[dname] = dh
    del models
    torch.cuda.empty_cache()
print("DATASET GENERALIZATION DONE")


===== B-enwik8 (seed 0) =====
[B-enwik8] step     1 | attention: 4.964 (ppl 143.2) gnorm 6.87 | v8a-delta: 4.974 (ppl 144.6) gnorm 4.35
[B-enwik8] step   250 | attention: 2.511 (ppl 12.3) gnorm 0.78 | v8a-delta: 2.098 (ppl 8.2) gnorm 0.97
[B-enwik8] step   500 | attention: 2.161 (ppl 8.7) gnorm 1.25 | v8a-delta: 1.825 (ppl 6.2) gnorm 1.15
[B-enwik8] step   750 | attention: 1.942 (ppl 7.0) gnorm 1.21 | v8a-delta: 1.696 (ppl 5.5) gnorm 0.99
[B-enwik8] step  1000 | attention: 1.807 (ppl 6.1) gnorm 1.29 | v8a-delta: 1.639 (ppl 5.1) gnorm 0.91
[B-enwik8] step  1250 | attention: 1.754 (ppl 5.8) gnorm 1.30 | v8a-delta: 1.573 (ppl 4.8) gnorm 0.85
[B-enwik8] step  1500 | attention: 1.683 (ppl 5.4) gnorm 1.26 | v8a-delta: 1.527 (ppl 4.6) gnorm 0.90
[B-enwik8] attention: final 1.683 best 1.683@1500
[B-enwik8] v8a-delta: final 1.527 best 1.527@1500
===== C-text8 (seed 0) =====
[C-text8] step     1 | attention: 2.970 (ppl 19.5) gnorm 7.50 | v8a-delta: 2.875 (ppl 17.7) gnorm 4.21
[C-text8] step   25

## 7. Zero-shot length generalization (T=256/512/1024/2048, seed-0 weights)

In [ ]:
# Fresh models built at SEQ_MAX=2048 (no rebuild needed), loaded with the
# T=128-trained seed-0 weights. Both models share the same (documented)
# position-embedding caveat past T=128, so the v8a-vs-attention ranking at
# each T stays apples-to-apples.
models_zs = build_models(vocabA, nlA, FFN["A-corpus"], seed=0)
models_zs["v8a-delta"].load_state_dict(torch.load("/content/v9a_seed0_v8a.pt"))
models_zs["attention"].load_state_dict(torch.load("/content/v9a_seed0_attn.pt"))
dA_full = DATA["A-corpus"][0]
zs = {128: {n: round(results[0][n]["final_val"], 3) for n in ("attention", "v8a-delta")}}
print("=== ZERO-SHOT LENGTH EVAL (val loss, no grad) ===")
for T_ in (256, 512, 1024, 2048):
    bs_e = 8 if T_ <= 512 else 4
    gb = make_get_batch(dA_full, T_, bs_e)
    row = {}
    for name, m in models_zs.items():
        try:
            row[name] = round(val_loss(m, gb, n=6), 3)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                row[name] = "OOM"
                torch.cuda.empty_cache()
            else:
                raise
    zs[T_] = row
    print(f"T={T_:5d}: " + " | ".join(f"{n}: {v}" for n, v in row.items()), flush=True)
print("ZERO-SHOT DONE")
del models_zs
torch.cuda.empty_cache()


=== ZERO-SHOT LENGTH EVAL (val loss, no grad) ===
T=  256: attention: 2.39 | v8a-delta: 1.305
T=  512: attention: 2.921 | v8a-delta: 1.291
T= 1024: attention: 3.16 | v8a-delta: 1.284
T= 2048: attention: 3.33 | v8a-delta: 1.296
ZERO-SHOT DONE


## 8. Train at T=512 (token-matched: 375 steps)

In [ ]:
# Token budget held constant: 375 steps x 32 x 512 == 1500 x 32 x 128.
# An attention OOM where v8a fits is recorded as a result, not a failure.
SEQ512, STEPS512 = 512, 375
gb512 = make_get_batch(dA, SEQ512, BS)
t512 = {}
for name in ("v8a-delta", "attention"):
    print(f"===== T=512 {name} =====", flush=True)
    try:
        ms = build_models(vocabA, nlA, FFN["A-corpus"], seed=0)
        m = ms[name]
        opt = torch.optim.AdamW(m.parameters(), lr=3e-4)
        for step in range(1, STEPS512 + 1):
            m.train()
            xb, yb = gb512("train")
            _, loss = m(xb, yb)
            opt.zero_grad(); loss.backward()
            gn = float(torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0))
            opt.step()
            check_finite(f"T512 {name} step {step}", [float(loss), gn])
            if step % 125 == 0 or step == 1:
                print(f"  step {step}: loss {float(loss):.3f} gnorm {gn:.2f}", flush=True)
        torch.cuda.reset_peak_memory_stats()
        m.train(); xb, yb = gb512("train")
        _, loss = m(xb, yb); m.zero_grad(); loss.backward()
        peak = torch.cuda.max_memory_allocated() / 1e6
        vl = val_loss(m, gb512)
        t512[name] = {"final_val": round(vl, 3), "vram_mb": round(peak)}
        print(f"T=512 {name}: final val {vl:.3f} peak {peak:.0f} MB", flush=True)
        del ms, m, opt
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            t512[name] = {"final_val": "OOM", "vram_mb": "OOM"}
            print(f"T=512 {name}: OUT OF MEMORY -- recorded as a result, not a failure", flush=True)
        else:
            raise
    torch.cuda.empty_cache()
print("T=512 TRAINING DONE")


===== T=512 v8a-delta =====
  step 1: loss 4.646 gnorm 4.94


/tmp/ipykernel_545/675161576.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  check_finite(f"T512 {name} step {step}", [float(loss), gn])


  step 125: loss 2.031 gnorm 1.34
  step 250: loss 1.675 gnorm 1.24
  step 375: loss 1.517 gnorm 1.00
T=512 v8a-delta: final val 1.478 peak 2796 MB
===== T=512 attention =====
  step 1: loss 4.807 gnorm 8.18
  step 125: loss 2.528 gnorm 0.53
  step 250: loss 2.453 gnorm 0.58
  step 375: loss 2.326 gnorm 1.57
T=512 attention: final val 2.380 peak 2414 MB
T=512 TRAINING DONE


## 9. Verdict — confirm or overturn (criteria from the brief §1a)

In [ ]:
import statistics
print("=== V9-A VERDICT ===")
v8 = [results[s]["v8a-delta"] for s in SEEDS]
at = [results[s]["attention"] for s in SEEDS]
def mean_std(xs):
    return sum(xs) / len(xs), statistics.stdev(xs) if len(xs) > 1 else 0.0
for key, label in [("final_val", "final val"), ("best_val", "best val"),
                   ("step250_val", "step-250 val")]:
    mv, sv = mean_std([r[key] for r in v8])
    ma, sa = mean_std([r[key] for r in at])
    pooled = math.sqrt((sv ** 2 + sa ** 2) / 2)
    print(f"{label}: v8a {mv:.3f}+/-{sv:.3f} | attn {ma:.3f}+/-{sa:.3f} | "
          f"margin {ma - mv:.3f} (2xpooled={2 * pooled:.3f})")
mv_f, sv_f = mean_std([r["final_val"] for r in v8])
ma_f, sa_f = mean_std([r["final_val"] for r in at])
pooled_f = math.sqrt((sv_f ** 2 + sa_f ** 2) / 2)
margin_f = ma_f - mv_f
flips = [s for s in SEEDS
         if results[s]["v8a-delta"]["final_val"] >= results[s]["attention"]["final_val"]]
print(f"seeds with v8a>=attn (ranking flips): {flips if flips else 'none'}")
confirm = (margin_f > 2 * pooled_f) and (mv_f + pooled_f < 1.339)
if confirm and not flips:
    print("VERDICT: CONFIRM -- seeded mean holds the gap to attention (>2x pooled std),")
    print("         the v8a band sits below V7's 1.339, no seed flips the ranking.")
else:
    print("VERDICT: OVERTURN (crown provisional) --")
    if flips:
        print(f"  - ranking flipped on seeds {flips}")
    if not (margin_f > 2 * pooled_f):
        print("  - seeded margin <= 2x pooled std (gap inside noise)")
    if not (mv_f + pooled_f < 1.339):
        print("  - v8a band not below V7's 1.339 (gap to V7 inside noise)")
    print("  V9 quality work is premature; scope claims to the current evidence.")
print("--- dataset scope findings ---")
for dname, dh in ds_results.items():
    for name in ("attention", "v8a-delta"):
        print(f"{dname} {name}: final {dh[name]['final_val']:.3f} best {dh[name]['best_val']:.3f}")
    lead = "v8a-delta" if dh["v8a-delta"]["final_val"] < dh["attention"]["final_val"] else "attention"
    flag = "  (RANKING FLIP -- crown is corpus-specific)" if lead == "attention" else ""
    print(f"  -> {dname} leader: {lead}{flag}")
print("--- length scope findings (zero-shot val) ---")
for T_ in sorted(zs):
    print(f"T={T_:5d}: " + " | ".join(f"{n}: {v}" for n, v in zs[T_].items()))
print("--- T=512 token-matched training ---")
for name, r in t512.items():
    print(f"{name}: {r}")
print("VERDICT DONE")


=== V9-A VERDICT ===
final val: v8a 1.312+/-0.008 | attn 1.402+/-0.024 | margin 0.090 (2xpooled=0.036)
best val: v8a 1.312+/-0.008 | attn 1.402+/-0.024 | margin 0.090 (2xpooled=0.036)
step-250 val: v8a 1.823+/-0.017 | attn 2.384+/-0.017 | margin 0.561 (2xpooled=0.034)
seeds with v8a>=attn (ranking flips): none
VERDICT: CONFIRM -- seeded mean holds the gap to attention (>2x pooled std),
         the v8a band sits below V7's 1.339, no seed flips the ranking.
--- dataset scope findings ---
B-enwik8 attention: final 1.683 best 1.683
B-enwik8 v8a-delta: final 1.527 best 1.527
  -> B-enwik8 leader: v8a-delta
C-text8 attention: final 1.519 best 1.519
C-text8 v8a-delta: final 1.507 best 1.507
  -> C-text8 leader: v8a-delta
--- length scope findings (zero-shot val) ---
T=  128: attention: 1.432 | v8a-delta: 1.324
T=  256: attention: 2.39 | v8a-delta: 1.305
T=  512: attention: 2.921 | v8a-delta: 1.291
T= 1024: attention: 3.16 | v8a-delta: 1.284
T= 2048: attention: 3.33 | v8a-delta: 1.296
--- T=5

## 10. Save everything (triple-redundant)

In [ ]:
lines = ["V9-A replication & generalization -- run record"]
for s in SEEDS:
    for name in ("attention", "v8a-delta"):
        r = results[s][name]
        lines.append(f"seed {s} {name}: final {r['final_val']:.3f} "
                     f"best {r['best_val']:.3f}@{r['best_step']} step250 {r['step250_val']:.3f}")
lines.append("--- datasets ---")
for dname, dh in ds_results.items():
    for name in ("attention", "v8a-delta"):
        lines.append(f"{dname} {name}: final {dh[name]['final_val']:.3f} best {dh[name]['best_val']:.3f}")
lines.append("--- zero-shot lengths ---")
for T_ in sorted(zs):
    lines.append(f"T={T_}: " + " ".join(f"{n}={v}" for n, v in zs[T_].items()))
lines.append("--- T=512 ---")
for name, r in t512.items():
    lines.append(f"T512 {name}: {r}")
open("/content/v9a_results.txt", "w").write("\n".join(lines))
print(open("/content/v9a_results.txt").read())
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    import shutil, datetime
    stamp = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
    shutil.copy("/content/v9a_results.txt",
                f"/content/drive/MyDrive/v9a_results_{stamp}.txt")
    print("Drive copy saved")
except Exception as e:
    print("Drive copy skipped:", e)
try:
    from google.colab import files
    files.download("/content/v9a_results.txt")
    print("download triggered")
except Exception as e:
    print("download skipped:", e)
print("SAVE DONE -- download the .ipynb via File > Download > .ipynb")


V9-A replication & generalization -- run record
seed 0 attention: final 1.432 best 1.432@1500 step250 2.396
seed 0 v8a-delta: final 1.324 best 1.324@1500 step250 1.837
seed 1 attention: final 1.365 best 1.365@1500 step250 2.362
seed 1 v8a-delta: final 1.311 best 1.311@1500 step250 1.818
seed 2 attention: final 1.404 best 1.404@1500 step250 2.391
seed 2 v8a-delta: final 1.311 best 1.311@1500 step250 1.842
seed 3 attention: final 1.406 best 1.406@1500 step250 2.371
seed 3 v8a-delta: final 1.302 best 1.302@1500 step250 1.800
seed 4 attention: final 1.404 best 1.404@1500 step250 2.400
seed 4 v8a-delta: final 1.312 best 1.312@1500 step250 1.818
--- datasets ---
B-enwik8 attention: final 1.683 best 1.683
B-enwik8 v8a-delta: final 1.527 best 1.527
C-text8 attention: final 1.519 best 1.519
C-text8 v8a-delta: final 1.507 best 1.507
--- zero-shot lengths ---
T=128: attention=1.432 v8a-delta=1.324
T=256: attention=2.39 v8a-delta=1.305
T=512: attention=2.921 v8a-delta=1.291
T=1024: attention=3.16 